In [ ]:
import duckdb, re

con = duckdb.connect("work.duckdb")
con.execute("SET temp_directory='duck_tmp'; SET memory_limit='10GB';")

# 1) load + remove 2024
con.execute("""
CREATE OR REPLACE TABLE base AS
SELECT d.authors, d.year, d.id, d.field_name, d.sub_name,
       d.cohort, d.career_age, d.period,
       coalesce(c.cite, 0) AS cite
FROM read_parquet('a2t_dec_final.parquet') d
LEFT JOIN (
    SELECT id, year, sum(cite) AS cite
    FROM read_parquet('cite_full.parquet')
    GROUP BY id, year
) c ON c.id = d.id AND c.year = d.year
""")

# 2) building blocks
con.execute("""
CREATE OR REPLACE TABLE panel AS          -- author-year unit
SELECT authors, year, any_value(cohort) AS cohort,
       any_value(career_age) AS career_age, any_value(period) AS period
FROM base GROUP BY authors, year;

CREATE OR REPLACE TABLE art AS            -- author-article
SELECT authors, id, year,
       any_value(field_name) AS field_name,
       any_value(cite) AS citations,
       bool_or(sub_name = 'Artificial Intelligence') AS is_ai
FROM base GROUP BY authors, id, year;

CREATE OR REPLACE TABLE co AS             -- coauthors = other authors on same article
SELECT DISTINCT a.authors, a.year, b.authors AS coauthor
FROM art a JOIN art b ON a.id = b.id AND a.authors <> b.authors;
""")

# 3) 7y window metrics (year-3 .. year+3)
con.execute("""
CREATE OR REPLACE TABLE w_art AS
SELECT p.authors, p.year,
       count(DISTINCT a.id) AS prod_7y,
       sum(a.citations)             AS cite_7y,
       avg(a.is_ai::INT)            AS share_ai_7y
FROM panel p
JOIN art a ON a.authors = p.authors AND a.year BETWEEN p.year-3 AND p.year+3
GROUP BY p.authors, p.year;

CREATE OR REPLACE TABLE w_co AS
SELECT p.authors, p.year, count(DISTINCT c.coauthor) AS n_unique_coauthors_7y
FROM panel p
LEFT JOIN co c ON c.authors = p.authors AND c.year BETWEEN p.year-3 AND p.year+3
GROUP BY p.authors, p.year;

CREATE OR REPLACE TABLE w_fc AS           -- articles per field in window
SELECT p.authors, p.year, a.field_name, count(*) AS n
FROM panel p
JOIN art a ON a.authors = p.authors AND a.year BETWEEN p.year-3 AND p.year+3
GROUP BY p.authors, p.year, a.field_name;

CREATE OR REPLACE TABLE w_gini AS
WITH r AS (
  SELECT *, row_number() OVER (PARTITION BY authors, year ORDER BY n) AS rk,
            count(*) OVER (PARTITION BY authors, year) AS k,
            sum(n)   OVER (PARTITION BY authors, year) AS tot
  FROM w_fc)
SELECT authors, year,
       2.0*sum(rk*n)/(any_value(k)*any_value(tot)) - (any_value(k)+1.0)/any_value(k) AS gini_field
FROM r GROUP BY authors, year;
""")

# 4) 26 field-share columns
fields = [r[0] for r in con.execute("SELECT DISTINCT field_name FROM base ORDER BY 1").fetchall()]
col = lambda f: "field_" + re.sub(r"\W+", "_", str(f).lower()).strip("_")
share_sql = ",\n".join(
    f"coalesce(sum(n) FILTER (WHERE field_name = '{str(f).replace(chr(39), chr(39)*2)}'), 0)::DOUBLE / sum(n) AS {col(f)}"
    for f in fields)
con.execute(f"""
CREATE OR REPLACE TABLE w_share AS
SELECT authors, year, {share_sql} FROM w_fc GROUP BY authors, year
""")

# 5) assemble + log(prod)
con.execute("""
CREATE OR REPLACE TABLE feat AS
SELECT p.*, a.prod_7y, a.cite_7y, a.share_ai_7y, c.n_unique_coauthors_7y,
       g.gini_field, ln(a.prod_7y) AS log_prod, s.* EXCLUDE (authors, year)
FROM panel p
JOIN w_art a   USING (authors, year)
JOIN w_co c    USING (authors, year)
JOIN w_gini g  USING (authors, year)
JOIN w_share s USING (authors, year)
""")

# 6) qcut-equivalent quantiles per period (duplicates='drop', labels 1..n_bins)
def qcut_sql(v, out):
    return f"""
    SELECT f.authors, f.year,
           1 + len(list_filter(q.e, x -> x < f.{v})) AS {out}
    FROM feat f
    JOIN (SELECT period,
                 list_filter(list_distinct(quantile_cont({v}, [0.2,0.4,0.6,0.8])),
                             x -> x > min({v}) AND x < max({v})) AS e
          FROM feat GROUP BY period) q USING (period)"""
con.execute(f"""
CREATE OR REPLACE TABLE quants AS
SELECT a.authors, a.year, a.quant_prod, b.quant_cite
FROM ({qcut_sql('prod_7y','quant_prod')}) a
JOIN ({qcut_sql('cite_7y','quant_cite')}) b USING (authors, year)
""")

# 7) previous occurrence + first quantiles
con.execute("""
CREATE OR REPLACE TABLE feat2 AS
SELECT f.*, q.quant_prod, q.quant_cite,
  lag(q.quant_prod)            OVER w  AS prev_quant_prod,
  lag(q.quant_cite)            OVER w  AS prev_quant_cite,
  lag(f.prod_7y)               OVER w  AS prev_prod_7y,
  lag(f.n_unique_coauthors_7y) OVER w  AS prev_coaut,
  first_value(q.quant_prod)    OVER w0 AS first_quant_prod,
  first_value(q.quant_cite)    OVER w0 AS first_quant_cite
FROM feat f JOIN quants q USING (authors, year)
WINDOW w  AS (PARTITION BY f.cohort, f.authors ORDER BY f.year),
       w0 AS (PARTITION BY f.cohort, f.authors ORDER BY f.year
              ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
""")

# 8) mean-centering (all numeric except ids, cohort, year, period) -> parquet
share_cols = [col(f) for f in fields]

keep = (
    ["authors", "year", "cohort", "period", "career_age",
     "prod_7y", "log_prod", "cite_7y", "share_ai_7y",
     "n_unique_coauthors_7y", "gini_field"]
    + share_cols
    + ["quant_prod", "quant_cite",
       "prev_quant_prod", "prev_quant_cite", "prev_prod_7y", "prev_coaut",
       "first_quant_prod", "first_quant_cite"]
)

# mean-center continuous variables only (field shares and quantiles stay raw)
continuous = ["career_age", "prod_7y", "log_prod", "cite_7y", "share_ai_7y",
              "n_unique_coauthors_7y", "gini_field",
              "prev_prod_7y", "prev_coaut"]

select_sql = ",\n".join(
    f"{c} - avg({c}) OVER () AS {c}" if c in continuous else c
    for c in keep
)

con.execute(f"""
COPY (SELECT {select_sql} FROM feat2)
TO 'a2t_final.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

IOException: IO Error: Could not write file "duck_tmp\duckdb_temp_storage_S96K-2.tmp" (error in WriteFile): Espace insuffisant sur le disque.


In [ ]:
import os
import duckdb, re
tmp = "C:/duck_tmp"            # replace with the drive that has the most free space
os.makedirs(tmp, exist_ok=True)
con = duckdb.connect("work.duckdb")

con.execute(f"SET temp_directory='{tmp}'; SET memory_limit='10GB'; SET preserve_insertion_order=false;")

for t in ["w_art", "w_fc", "w_gini", "w_share", "co", "w_co", "feat", "feat2", "quants"]:
    con.execute(f"DROP TABLE IF EXISTS {t}")
con.execute("CHECKPOINT")

# step 1 (base) stays the same
con.execute("""
CREATE OR REPLACE TABLE base AS
SELECT d.authors, d.year, d.id, d.field_name, d.sub_name,
       d.cohort, d.career_age, d.period, d.gender, d.insti_rank,
       coalesce(c.cite, 0) AS cite
FROM read_parquet('a2t_dec_final.parquet') d
LEFT JOIN (
    SELECT id, year, sum(cite) AS cite
    FROM read_parquet('cite_full.parquet')
    GROUP BY id, year
) c ON c.id = d.id AND c.year = d.year
""")

fields = [r[0] for r in con.execute("SELECT DISTINCT field_name FROM base ORDER BY 1").fetchall()]
col = lambda f: "field_" + re.sub(r"\W+", "_", str(f).lower()).strip("_")
share_cols = [col(f) for f in fields]
esc = lambda f: str(f).replace("'", "''")

# 2) building blocks
con.execute("""
CREATE OR REPLACE TABLE panel AS
SELECT authors, year, any_value(cohort) AS cohort,
       any_value(career_age) AS career_age, any_value(period) AS period
FROM base GROUP BY authors, year;

CREATE OR REPLACE TABLE art AS
SELECT authors, id, year,
       any_value(field_name) AS field_name,
       any_value(cite) AS citations,
       bool_or(sub_name = 'Artificial Intelligence') AS is_ai
FROM base GROUP BY authors, id, year;
""")

# 3) author-year counts, then 7y window sums with a RANGE frame (no join)
cnt = ",\n".join(f"sum((field_name = '{esc(f)}')::INT) AS c_{i}" for i, f in enumerate(fields))
wsum = ",\n".join(f"sum(c_{i}) OVER win AS w_{i}" for i in range(len(fields)))
con.execute(f"""
CREATE OR REPLACE TABLE ay AS
SELECT authors, year, count(*) AS n_art, sum(citations) AS cite,
       sum(is_ai::INT) AS n_ai, {cnt}
FROM art GROUP BY authors, year
""")
con.execute(f"""
CREATE OR REPLACE TABLE w_all AS
SELECT authors, year,
       sum(n_art) OVER win AS prod_7y,
       sum(cite)  OVER win AS cite_7y,
       sum(n_ai)  OVER win AS n_ai_7y,
       {wsum}
FROM ay
WINDOW win AS (PARTITION BY authors ORDER BY year RANGE BETWEEN 3 PRECEDING AND 3 FOLLOWING)
""")

# 4) shares + gini from the window counts (gini over fields present, sorted ascending)
lst = "list_sort(list_filter([" + ",".join(f"w_{i}" for i in range(len(fields))) + "], x -> x > 0))"
share_sql = ",\n".join(f"w_{i}::DOUBLE / prod_7y AS {col(f)}" for i, f in enumerate(fields))
con.execute(f"""
CREATE OR REPLACE TABLE w_gs AS
SELECT authors, year, prod_7y, cite_7y,
       n_ai_7y::DOUBLE / prod_7y AS share_ai_7y,
       2.0 * list_sum(list_transform(l, (x, i) -> x * i)) / (len(l) * prod_7y)
           - (len(l) + 1.0) / len(l) AS gini_field,
       {share_sql}
FROM (SELECT *, {lst} AS l FROM w_all)
""")

# 5) unique coauthors in window, batched by author hash (only step that needs a join)
NB = 20
con.execute("CREATE OR REPLACE TABLE w_co (authors VARCHAR, year INT, n_unique_coauthors_7y BIGINT)")
# ^ adjust VARCHAR/INT to your real types of authors/year
for k in range(NB):
    con.execute(f"""
    INSERT INTO w_co
    WITH a AS (SELECT * FROM art WHERE hash(authors) % {NB} = {k}),
    co AS (SELECT DISTINCT a.authors, a.year, b.authors AS coauthor
           FROM a JOIN art b ON a.id = b.id AND a.authors <> b.authors),
    p AS (SELECT authors, year FROM panel WHERE hash(authors) % {NB} = {k})
    SELECT p.authors, p.year, count(DISTINCT c.coauthor)
    FROM p LEFT JOIN co c ON c.authors = p.authors AND c.year BETWEEN p.year-3 AND p.year+3
    GROUP BY p.authors, p.year
    """)

# assemble (replaces old step 5)
con.execute("""
CREATE OR REPLACE TABLE feat AS
SELECT p.*, g.prod_7y, g.cite_7y, g.share_ai_7y, c.n_unique_coauthors_7y,
       g.gini_field, ln(g.prod_7y) AS log_prod, g.* EXCLUDE (authors, year, prod_7y, cite_7y, share_ai_7y, gini_field)
FROM panel p
JOIN w_gs g USING (authors, year)
JOIN w_co c USING (authors, year)
""")
con.execute("DROP TABLE w_all; DROP TABLE w_gs; DROP TABLE ay; CHECKPOINT")
# 6) qcut-equivalent quantiles per period (duplicates='drop', labels 1..n_bins)
def qcut_sql(v, out):
    return f"""
    SELECT f.authors, f.year,
           1 + len(list_filter(q.e, x -> x < f.{v})) AS {out}
    FROM feat f
    JOIN (SELECT period,
                 list_filter(list_distinct(quantile_cont({v}, [0.2,0.4,0.6,0.8])),
                             x -> x > min({v}) AND x < max({v})) AS e
          FROM feat GROUP BY period) q USING (period)"""
con.execute(f"""
CREATE OR REPLACE TABLE quants AS
SELECT a.authors, a.year, a.quant_prod, b.quant_cite
FROM ({qcut_sql('prod_7y','quant_prod')}) a
JOIN ({qcut_sql('cite_7y','quant_cite')}) b USING (authors, year)
""")

# 7) previous occurrence + first quantiles
con.execute("""
CREATE OR REPLACE TABLE feat2 AS
SELECT f.*, q.quant_prod, q.quant_cite,
  lag(q.quant_prod)            OVER w  AS prev_quant_prod,
  lag(q.quant_cite)            OVER w  AS prev_quant_cite,
  lag(f.prod_7y)               OVER w  AS prev_prod_7y,
  lag(f.n_unique_coauthors_7y) OVER w  AS prev_coaut,
  first_value(q.quant_prod)    OVER w0 AS first_quant_prod,
  first_value(q.quant_cite)    OVER w0 AS first_quant_cite
FROM feat f JOIN quants q USING (authors, year)
WINDOW w  AS (PARTITION BY f.cohort, f.authors ORDER BY f.year),
       w0 AS (PARTITION BY f.cohort, f.authors ORDER BY f.year
              ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING)
""")

# 8) mean-centering (all numeric except ids, cohort, year, period) -> parquet
share_cols = [col(f) for f in fields]

keep = (
    ["authors", "year", "cohort", "period", "career_age",
     "prod_7y", "log_prod", "cite_7y", "share_ai_7y",
     "n_unique_coauthors_7y", "gini_field","gender","insti_rank"]
    + share_cols
    + ["quant_prod", "quant_cite",
       "prev_quant_prod", "prev_quant_cite", "prev_prod_7y", "prev_coaut",
       "first_quant_prod", "first_quant_cite"]
)

# mean-center continuous variables only (field shares and quantiles stay raw)
continuous = ["career_age", "prod_7y", "log_prod", "cite_7y", "share_ai_7y",
              "n_unique_coauthors_7y", "gini_field",
              "prev_prod_7y", "prev_coaut"]

select_sql = ",\n".join(
    f"{c} - avg({c}) OVER () AS {c}" if c in continuous else c
    for c in keep
)

con.execute(f"""
COPY (SELECT {select_sql} FROM feat2)
TO 'a2t_final.parquet' (FORMAT PARQUET, COMPRESSION ZSTD)
""")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

...

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Insti_rank (from column institution and citation)

In [ ]:
a2t["institution"] = a2t["institution"].replace("NONE",None)
a2t["insti_rank"] = a2t.groupby(["institution"])["cite"].transform("mean")
a2t["insti_rank"] = pd.qcut(a2t["insti_rank"], 5, labels=range(1,6))